# 02. Sentiment Modeling: Detecting Information Asymmetry

**Context**: In the Kenyan e-commerce landscape, numerical star ratings often mask qualitative dissatisfaction. This notebook builds an NLP pipeline to identify "Information Asymmetry"—cases where products maintain high ratings despite critical feedback hidden in local dialects (Sheng/Swahili).

## 2.1 Load Preprocessed Data

We pull the synthesized NLP dataset created in Phase 1. This dataset already includes slang-mapped tokens and joined product metadata.

In [5]:
import sys

sys.path.append("..")

import pandas as pd
import numpy as np
from src.data_preprocessing import DataLoader, TextPreprocessor
from src.sentiment_analysis import SentimentModel

# 1. Initialize tools
loader = DataLoader(data_dir="../data/raw/")
text_proc = TextPreprocessor()

# 2. Build the Base NLP Frame first
# (This creates the nlp_df we need for the merge)
_, prods, revs = loader.load_all()
nlp_df = text_proc.build_nlp_frame(revs, prods)

# 3. Load and Clean the New Jumia Egypt metadata
egypt_prods = pd.read_csv(
    "../data/raw/jumia_products.csv",
    encoding="latin1",
)

# Extract numeric price and rating
egypt_prods["price_numeric"] = egypt_prods["price"].apply(text_proc.parse_price)
egypt_prods["official_rating"] = (
    egypt_prods["avg_rate"].str.extract(r"(\d+\.\d+|\d+)").astype(float)
)

# Clean reviews_count: "46 verified ratings" -> 46
egypt_prods["reviews_count_numeric"] = (
    egypt_prods["reviews_count"].str.extract(r"(\d+)").fillna(0).astype(int)
)

# 4. Enrich the NLP Frame
# We join the 'Official' Jumia metrics to our 'Text-based' sentiment data
enriched_nlp = nlp_df.merge(
    egypt_prods[["id", "official_rating", "reviews_count_numeric", "price_numeric"]],
    left_on="sku",
    right_on="id",
    how="left",
)

print(f"Enriched NLP Frame Ready: {enriched_nlp.shape[0]} samples")
print(f"Columns available: {enriched_nlp.columns.tolist()}")

Enriched NLP Frame Ready: 92 samples
Columns available: ['product', 'sku', 'rating', 'title', 'review', 'date', 'customer', 'product_id', 'product_name', 'product_category', 'price_numeric_x', 'sentiment_target', 'full_text', 'tokens', 'id', 'official_rating', 'reviews_count_numeric', 'price_numeric_y']


## 2.2 Vectorization & Training

We use TF-IDF (Term Frequency-Inverse Document Frequency) to convert our tokens into a numerical matrix. By using ngram_range=(1, 2), we capture both individual words (e.g., "feki") and phrases (e.g., "not good").

In [6]:
# Initialize our model class from src/sentiment_analysis.py
sm = SentimentModel(max_features=2500)

# 1. Transform text to features
X_tfidf = sm.prepare_features(nlp_df["tokens"])
y = nlp_df["sentiment_target"]

# 2. Train-Test Split (Stratified to maintain sentiment balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Fit the Classifier
sm.train(X_train, y_train)

# 4. Evaluation (This will print the report)
sm.evaluate(X_test, y_test)

--- Sentiment Engine: Detailed Report ---
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         1
         1.0       0.95      1.00      0.97        18

    accuracy                           0.95        19
   macro avg       0.47      0.50      0.49        19
weighted avg       0.90      0.95      0.92        19



array([[ 0,  1],
       [ 0, 18]])

## 2.3 Model Persistence

To use this model in our final integration (Phase 4), we must persist the trained objects to the models/ directory.

In [7]:
# This creates 'sentiment_rf.joblib' and 'tfidf_vec.joblib' in /models
sm.save()

☑ Success: Sentiment components saved to ../models/sentiment_rf.joblib and ../models/tfidf_vec.joblib
